In [1]:
!pip install opencv-python yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 48.8 MB/s eta 0:00:00


In [2]:
!pip install opencv-python-headless moviepy

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 1182756-hd_1920_1080_25fps.mp4 to 1182756-hd_1920_1080_25fps.mp4
Saving 8322694-uhd_4096_2160_25fps.mp4 to 8322694-uhd_4096_2160_25fps.mp4


In [3]:
import cv2
import numpy as np
from google.colab import files
from moviepy.editor import VideoFileClip

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



In [4]:
uploaded = files.upload()

Saving 1182756-hd_1920_1080_25fps.mp4 to 1182756-hd_1920_1080_25fps.mp4
Saving 8322694-uhd_4096_2160_25fps.mp4 to 8322694-uhd_4096_2160_25fps.mp4


In [5]:
def process_video(video_path, output_name):

    cap = cv2.VideoCapture(video_path)

    ret, frame1 = cap.read()
    frame1 = cv2.resize(frame1,(1280,720))
    prvs = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

    hsv = np.zeros_like(frame1)
    hsv[...,1] = 255

    out = cv2.VideoWriter(output_name,
                          cv2.VideoWriter_fourcc(*'mp4v'),
                          30,
                          (1280,720))

    while True:

        ret, frame2 = cap.read()
        if not ret:
            break

        frame2 = cv2.resize(frame2,(1280,720))
        next = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

        flow = cv2.calcOpticalFlowFarneback(
            prvs,next,None,
            0.5,3,15,3,5,1.2,0)

        mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])

        hsv[...,0] = ang*180/np.pi/2
        hsv[...,2] = cv2.normalize(mag,None,0,255,cv2.NORM_MINMAX)

        rgb = cv2.cvtColor(hsv,cv2.COLOR_HSV2BGR)

        out.write(rgb)

        prvs = next

    cap.release()
    out.release()

In [6]:
process_video("blind_man.mp4","blind_man_flow.mp4")
process_video("puppy.mp4","puppy_flow.mp4")

In [7]:
def track_points(video):

    cap = cv2.VideoCapture(video)

    ret, frame1 = cap.read()
    frame1 = cv2.resize(frame1,(1280,720))
    old_gray = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

    p0 = cv2.goodFeaturesToTrack(old_gray,
                                 maxCorners=100,
                                 qualityLevel=0.3,
                                 minDistance=7)

    ret, frame2 = cap.read()
    frame2 = cv2.resize(frame2,(1280,720))
    frame_gray = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    p1, st, err = cv2.calcOpticalFlowPyrLK(
        old_gray,frame_gray,p0,None)

    good_new = p1[st==1]
    good_old = p0[st==1]

    for i,(new,old) in enumerate(zip(good_new,good_old)):

        a,b = new.ravel()
        c,d = old.ravel()

        print("Old:",(c,d)," -> New:",(a,b))

In [8]:
track_points("blind_man.mp4")
track_points("puppy.mp4")

Old: (np.float32(432.0), np.float32(13.0))  -> New: (np.float32(431.61713), np.float32(12.917659))
Old: (np.float32(959.0), np.float32(9.0))  -> New: (np.float32(958.62646), np.float32(8.668114))
Old: (np.float32(1123.0), np.float32(116.0))  -> New: (np.float32(1122.6995), np.float32(115.63844))
Old: (np.float32(1049.0), np.float32(158.0))  -> New: (np.float32(1048.6929), np.float32(157.64238))
Old: (np.float32(976.0), np.float32(7.0))  -> New: (np.float32(975.67255), np.float32(6.671233))
Old: (np.float32(92.0), np.float32(139.0))  -> New: (np.float32(91.62742), np.float32(139.09856))
Old: (np.float32(1127.0), np.float32(110.0))  -> New: (np.float32(1126.6847), np.float32(109.61986))
Old: (np.float32(339.0), np.float32(2.0))  -> New: (np.float32(338.63266), np.float32(1.9829111))
Old: (np.float32(433.0), np.float32(26.0))  -> New: (np.float32(432.6526), np.float32(25.903334))
Old: (np.float32(233.0), np.float32(24.0))  -> New: (np.float32(232.62589), np.float32(24.03506))
Old: (np.flo

In [12]:
import cv2
import numpy as np

def visualize_tracking(video):

    cap = cv2.VideoCapture(video)

    ret, frame1 = cap.read()
    frame1 = cv2.resize(frame1,(1280,720))
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

    p0 = cv2.goodFeaturesToTrack(gray1,
                                 maxCorners=100,
                                 qualityLevel=0.3,
                                 minDistance=7)

    ret, frame2 = cap.read()
    frame2 = cv2.resize(frame2,(1280,720))
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    p1, st, err = cv2.calcOpticalFlowPyrLK(gray1,gray2,p0,None)

    good_new = p1[st==1]
    good_old = p0[st==1]

    for new, old in zip(good_new,good_old):

        a,b = new.ravel().astype(int)
        c,d = old.ravel().astype(int)

        cv2.arrowedLine(frame2,(c,d),(a,b),(0,255,0),2)
        cv2.circle(frame2,(a,b),3,(0,0,255),-1)

    cv2.imwrite("tracking_result1.png",frame2)

#visualize_tracking("blind_man.mp4")
visualize_tracking("puppy.mp4")

In [13]:
from google.colab import files
files.download("tracking_result1.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>